### **Data pre-processing**


In [1]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

# ===================== PATHS =====================
input_root = Path(r"C:\Users\aymna\Desktop\Onedrive\OneDrive - Umm Al-Qura University\Data of graduation project\Data\المنطقة الغربية\clean Google map")
output_root = Path(r"C:\Users\aymna\Desktop\Onedrive\OneDrive - Umm Al-Qura University\Data of graduation project\Data\المنطقة الغربية\After Cleaning v3")
output_root.mkdir(parents=True, exist_ok=True)

# ===================== SETTINGS =====================
TEXT_COL = "text"
MIN_WORDS = 2

# Source columns in your files
RAW_COL = "Text"
TR_COL = "textTranslated"  # Arabic translation

# Transformer settings (lighter cleaning)
REMOVE_LATIN_TR = True
KEEP_DIGITS_TR = True

# ML settings
REMOVE_LATIN_ML = True
KEEP_DIGITS_ML = True   # keep numbers/dates
REMOVE_STOPWORDS_ML = True
BIND_NEGATION_ML = True  # ✅ recommended for sentiment with TF-IDF

# Add emoji sentiment token to ML text
ADD_EMO_TOKEN_TO_ML = True  # ✅ recommended

# ===================== REGEX PATTERNS =====================
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
TATWEEL_RE = re.compile(r"\u0640")
DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0657-\u065F\u0670\u06D6-\u06ED]")
PUNCT_SYMBOLS_RE = re.compile(r"[^\w\s\u0600-\u06FF]")  # keep Arabic letters + digits + underscore + spaces
MULTISPACE_RE = re.compile(r"\s+")
LATIN_RE = re.compile(r"[A-Za-z]+")
DIGITS_RE = re.compile(r"\d+")

# Emoji matcher (single emoji tokens; important: NO '+')
EMOJI_RE = re.compile(
    "[" +
    "\U0001F300-\U0001FAFF" +
    "\U00002700-\U000027BF" +
    "\U00002600-\U000026FF" +
    "]",
    flags=re.UNICODE
)

# IMPORTANT: compress repeated ARABIC LETTERS only (does NOT touch digits like 5000)
AR_LETTER_REPEATS = re.compile(r"([\u0600-\u06FF])\1{2,}")

# ===================== EMOJI SENTIMENT (expand later) =====================
POS_EMOJI = set(list("😀😃😄😁😆😊😍🥰😘😇🙂😉🤩😎😺😸😹👍👏🙌💯🔥⭐️🌟✨❤️🩵💚💛💜🤍"))
NEG_EMOJI = set(list("😞😔😟😕🙁☹️😣😖😫😩😢😭😤😠😡🤬👎💔💩🤢🤮😒😓😥😰😨😱"))

# ===================== STOPWORDS (sentiment-safe) =====================
NEGATION_WORDS = set("ما لا لم لن ليس مو مش بدون".split())
INTENSIFIERS = set("جدا جدًا مره مرة كثير كتير للغاية للغايه جدًاا جدااa".split())

AR_STOPWORDS = set("""
في من على إلى عن هذا هذه ذلك تلك هناك هنا كان كانت يكون تكون
مع أو ثم حيث الذي التي الذين اللواتي اذا إذ قد كل بعض أيضا فقط حتى بعد قبل عند بين
انه انها هم هن نحن انت انتم انا
""".split())

AR_STOPWORDS = AR_STOPWORDS - NEGATION_WORDS - INTENSIFIERS

# ===================== HELPERS =====================
def normalize_unicode(text: str) -> str:
    # ✅ prevent <NA> / nan turning into text
    if pd.isna(text):
        return ""
    return unicodedata.normalize("NFKC", str(text))

def remove_noise(text: str) -> str:
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)  # keep hashtag word without '#'
    return text

def arabic_normalize_light(text: str) -> str:
    text = DIACRITICS_RE.sub("", text)
    text = TATWEEL_RE.sub("", text)
    text = re.sub(r"[إأٱآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    return text

def remove_punct(text: str) -> str:
    return PUNCT_SYMBOLS_RE.sub(" ", text)

def remove_emojis_from_text(text: str) -> str:
    return EMOJI_RE.sub(" ", text)

def squeeze_repeats_safe(text: str) -> str:
    return AR_LETTER_REPEATS.sub(r"\1", text)

def finalize(text: str) -> str:
    return MULTISPACE_RE.sub(" ", str(text)).strip()

def extract_emojis(text: str):
    if pd.isna(text):
        return []
    return EMOJI_RE.findall(str(text))

def emoji_counts(emoji_list):
    if not emoji_list:
        return 0, 0
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    return pos, neg

def emoji_sentiment(emoji_list):
    if not emoji_list:
        return "NEU"
    pos = sum(e in POS_EMOJI for e in emoji_list)
    neg = sum(e in NEG_EMOJI for e in emoji_list)
    if pos > neg:
        return "POS"
    if neg > pos:
        return "NEG"
    return "NEU"

def emoji_score(pos_count: int, neg_count: int) -> int:
    return int(pos_count - neg_count)

def bind_negation(text: str) -> str:
    toks = text.split()
    out = []
    i = 0
    while i < len(toks):
        if toks[i] in NEGATION_WORDS and i + 1 < len(toks):
            out.append(toks[i] + "_" + toks[i + 1])
            i += 2
        else:
            out.append(toks[i])
            i += 1
    return " ".join(out)

def tr_cleanup(text: str) -> str:
    if REMOVE_LATIN_TR:
        text = LATIN_RE.sub(" ", text)
    if not KEEP_DIGITS_TR:
        text = DIGITS_RE.sub(" ", text)
    return finalize(text)

def ml_cleanup(text: str) -> str:
    if REMOVE_LATIN_ML:
        text = LATIN_RE.sub(" ", text)
    if not KEEP_DIGITS_ML:
        text = DIGITS_RE.sub(" ", text)

    text = finalize(text)

    if BIND_NEGATION_ML:
        text = bind_negation(text)

    if REMOVE_STOPWORDS_ML:
        toks = [t for t in text.split() if (t not in AR_STOPWORDS) and (len(t) > 1)]
        text = " ".join(toks)

    return finalize(text)

def wc(s: str) -> int:
    s = str(s).strip()
    return 0 if not s else len(s.split())

# ===================== BATCH PROCESS =====================
processed = 0
failed = 0
changed_numbers_total_files = 0

for file in input_root.rglob("*.xlsx"):
    try:
        df = pd.read_excel(file)

        # ===================== DROP NON-ESSENTIAL COLUMNS =====================
        cat_cols = [c for c in df.columns if str(c).startswith("categories/")]
        drop_cols = cat_cols + [c for c in ["countryCode", "name"] if c in df.columns]
        df.drop(columns=drop_cols, inplace=True, errors="ignore")

        # ===================== ENSURE SOURCE COLS EXIST =====================
        if RAW_COL not in df.columns:
            df[RAW_COL] = pd.NA
        if TR_COL not in df.columns:
            df[TR_COL] = pd.NA

        # Keep original raw text
        df["Text_Orig"] = df[RAW_COL]

        # ===================== BUILD UNIFIED TEXT (NO NaN / NO "nan") =====================
        df[TEXT_COL] = (
            df[TR_COL]
              .combine_first(df[RAW_COL])  # TR first then RAW
              .fillna("")                  # no NaN
              .astype(str)
        )

        # Prevent literal "nan" / "None" becoming text
        df[TEXT_COL] = df[TEXT_COL].replace({"nan": "", "None": "", "<NA>": ""})

        # Drop translation column if you want
        df.drop(columns=[TR_COL], inplace=True, errors="ignore")

        raw = df[TEXT_COL]  # already safe

        # ---- Emoji features ----
        df["Emoji_List"] = raw.apply(extract_emojis)
        df["Emoji_Count"] = df["Emoji_List"].apply(len)

        df["Emoji_Pos_Count"], df["Emoji_Neg_Count"] = zip(*df["Emoji_List"].apply(emoji_counts))
        df["Emoji_Score"] = df.apply(lambda r: emoji_score(r["Emoji_Pos_Count"], r["Emoji_Neg_Count"]), axis=1)

        df["Emoji_Sentiment"] = df["Emoji_List"].apply(emoji_sentiment)

        # ---- Base clean ----
        df["Text_Base"] = (
            raw.apply(normalize_unicode)
            .apply(remove_noise)
            .apply(remove_emojis_from_text)   # emojis stored in Emoji_List
            .apply(arabic_normalize_light)
            .apply(remove_punct)
            .apply(squeeze_repeats_safe)
            .apply(finalize)
        )

        # ---- Numeric integrity check (internal only) ----
        nums_raw = raw.apply(lambda s: DIGITS_RE.findall(str(s)))
        nums_base = df["Text_Base"].apply(lambda s: DIGITS_RE.findall(str(s)))
        changed_numbers_rows = (nums_raw != nums_base).sum()
        if changed_numbers_rows > 0:
            changed_numbers_total_files += 1
            print(f"⚠️ Numbers changed in file: {file.name} | rows affected: {changed_numbers_rows}")

        # ---- Two outputs ----
        df["Text_TR"] = df["Text_Base"].apply(tr_cleanup)
        df["Text_ML"] = df["Text_Base"].apply(ml_cleanup)

        # ---- Add emoji sentiment token to ML text ----
        if ADD_EMO_TOKEN_TO_ML:
            emo_token = df["Emoji_Sentiment"].map({"POS": "EMO_POS", "NEG": "EMO_NEG", "NEU": "EMO_NEU"}).fillna("EMO_NEU")
            df["Text_ML"] = (df["Text_ML"] + " " + emo_token).apply(finalize)

        # ---- Short flags ----
        df["Is_Short_TR"] = df["Text_TR"].apply(wc) < MIN_WORDS
        df["Is_Short_ML"] = df["Text_ML"].apply(wc) < MIN_WORDS

        # ---- Save output preserving folder structure ----
        rel = file.relative_to(input_root)
        out_file = output_root / rel.parent / f"{file.stem}_textready{file.suffix}"
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_excel(out_file, index=False)

        processed += 1
        print(f"✅ Saved: {out_file}")

    except Exception as e:
        failed += 1
        print(f"❌ Failed: {file.name} | {e}")

print(f"\n🎉 DONE | Processed: {processed} | Failed: {failed} | Files with number-changes warnings: {changed_numbers_total_files}")


✅ Saved: C:\Users\aymna\Desktop\Onedrive\OneDrive - Umm Al-Qura University\Data of graduation project\Data\المنطقة الغربية\After Cleaning v3\الشعيبة\adjusted_السفينة الغارقة الشعيبة_final_textready.xlsx
✅ Saved: C:\Users\aymna\Desktop\Onedrive\OneDrive - Umm Al-Qura University\Data of graduation project\Data\المنطقة الغربية\After Cleaning v3\الشعيبة\adjusted_مرسى القطان_final_textready.xlsx
✅ Saved: C:\Users\aymna\Desktop\Onedrive\OneDrive - Umm Al-Qura University\Data of graduation project\Data\المنطقة الغربية\After Cleaning v3\الشعيبة\بحر الشعيبة _final_textready.xlsx
✅ Saved: C:\Users\aymna\Desktop\Onedrive\OneDrive - Umm Al-Qura University\Data of graduation project\Data\المنطقة الغربية\After Cleaning v3\الشعيبة\بحر الشعيبة منطقة السباحة _final_textready.xlsx
⚠️ Numbers changed in file: كورنيش السيف_final.xlsx | rows affected: 1
✅ Saved: C:\Users\aymna\Desktop\Onedrive\OneDrive - Umm Al-Qura University\Data of graduation project\Data\المنطقة الغربية\After Cleaning v3\الشعيبة\كورنيش

## **Checking The Proccesing Work** 

### **Imports**

In [ ]:
import pandas as pd
import numpy as np
import re

raw_path   = r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\Google Maps Data\منطقة الباحة\منتزة غابة رغدان.xlsx"
ready_path = r"C:\Users\aws12\Desktop\Pre-Proccesing Step -GP 2\After Cleaning v3\منطقة الباحة\منتزة غابة رغدان_textready.xlsx"

raw_df = pd.read_excel(raw_path)
ready_df = pd.read_excel(ready_path)

print("RAW:", raw_df.shape)
print("READY:", ready_df.shape)


RAW: (37938, 16)
READY: (37938, 23)


### **Show sample data**


In [ ]:
display(raw_df.head(3))
display(ready_df.head(3))

print("\nRAW columns:\n", list(raw_df.columns))
print("\nREADY columns:\n", list(ready_df.columns))


,categories/0,categories/1,categories/2,categoryName,city,countryCode,location/lat,location/lng,name,neighborhood,stars,publishedAtDate,text,textTranslated,title,street
0,متنزه,غابة قومية,مزار سياحي,متنزه,الباحة,SA,20.020606,41.432788,Waheed Khalid,NaN,5,2025-10-27T14:56:49.177Z,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905
1,متنزه,غابة قومية,مزار سياحي,متنزه,الباحة,SA,20.020606,41.432788,سلطان المالكي,NaN,5,2025-10-27T09:01:04.964Z,NaN,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905
2,متنزه,غابة قومية,مزار سياحي,متنزه,الباحة,SA,20.020606,41.432788,mrdi qarni,NaN,5,2025-10-26T01:13:00.119Z,الاول,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905


,categoryName,city,location/lat,location/lng,neighborhood,stars,publishedAtDate,text,title,street,...,Emoji_Count,Emoji_Pos_Count,Emoji_Neg_Count,Emoji_Score,Emoji_Sentiment,Text_Base,Text_TR,Text_ML,Is_Short_TR,Is_Short_ML
0,متنزه,الباحة,20.020606,41.432788,NaN,5,2025-10-27T14:56:49.177Z,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,منتزه غابة رغدان,4657 الحكم الزروقي، 6905,...,0,0,0,0,NEU,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,تم تطويره عن السابق و الشيء المميز لا يوجد قرو...,تم تطويره السابق الشيء المميز لا_يوجد قرود الم...,False,False
1,متنزه,الباحة,20.020606,41.432788,NaN,5,2025-10-27T09:01:04.964Z,NaN,منتزه غابة رغدان,4657 الحكم الزروقي، 6905,...,0,0,0,0,NEU,NaN,NaN,EMO_NEU,True,True
2,متنزه,الباحة,20.020606,41.432788,NaN,5,2025-10-26T01:13:00.119Z,الاول,منتزه غابة رغدان,4657 الحكم الزروقي، 6905,...,0,0,0,0,NEU,الاول,الاول,الاول EMO_NEU,True,False



RAW columns:
 ['categories/0', 'categories/1', 'categories/2', 'categoryName', 'city', 'countryCode', 'location/lat', 'location/lng', 'name', 'neighborhood', 'stars', 'publishedAtDate', 'text', 'textTranslated', 'title', 'street']

READY columns:
 ['categoryName', 'city', 'location/lat', 'location/lng', 'neighborhood', 'stars', 'publishedAtDate', 'text', 'title', 'street', 'Text_Orig', 'Text', 'Emoji_List', 'Emoji_Count', 'Emoji_Pos_Count', 'Emoji_Neg_Count', 'Emoji_Score', 'Emoji_Sentiment', 'Text_Base', 'Text_TR', 'Text_ML', 'Is_Short_TR', 'Is_Short_ML']


### **Table of deleted and added columns**


In [ ]:
raw_cols = set(raw_df.columns.astype(str))
ready_cols = set(ready_df.columns.astype(str))

dropped = sorted(list(raw_cols - ready_cols))
added   = sorted(list(ready_cols - raw_cols))

col_changes = pd.DataFrame({
    "Dropped Columns (اختفت)": pd.Series(dropped),
    "Added Columns (انضافت)": pd.Series(added)
})

col_changes


,Dropped Columns (اختفت),Added Columns (انضافت)
0,categories/0,Emoji_Count
1,categories/1,Emoji_List
2,categories/2,Emoji_Neg_Count
3,countryCode,Emoji_Pos_Count
4,name,Emoji_Score
5,textTranslated,Emoji_Sentiment
6,NaN,Is_Short_ML
7,NaN,Is_Short_TR
8,NaN,Text
9,NaN,Text_Base


### **Summary of the number of records and columns (before/after)**


In [ ]:
overview = pd.DataFrame({
    "Dataset": ["Before (RAW)", "After (READY)"],
    "Rows": [len(raw_df), len(ready_df)],
    "Columns": [len(raw_df.columns), len(ready_df.columns)]
})
overview


,Dataset,Rows,Columns
0,Before (RAW),37938,16
1,After (READY),37938,23


### **Percentage of empty text before/after**


In [ ]:
def empty_ratio(series):
    s = series.fillna("").astype(str).str.strip()
    return round(((s == "") | (s.str.lower().isin(["nan","none"]))).mean()*100, 2)

# أعمدة النص حسب ملفاتكم
raw_text_col = "text" if "text" in raw_df.columns else None
raw_tr_col   = "textTranslated" if "textTranslated" in raw_df.columns else None

ready_text_col = "Text" if "Text" in ready_df.columns else None
base_col = "Text_Base" if "Text_Base" in ready_df.columns else None
ml_col   = "Text_ML" if "Text_ML" in ready_df.columns else None
tr_col   = "Text_TR" if "Text_TR" in ready_df.columns else None

summary_empty = pd.DataFrame({
    "Column": [raw_text_col, raw_tr_col, ready_text_col, base_col, ml_col, tr_col],
    "Empty_%": [
        empty_ratio(raw_df[raw_text_col]) if raw_text_col else np.nan,
        empty_ratio(raw_df[raw_tr_col]) if raw_tr_col else np.nan,
        empty_ratio(ready_df[ready_text_col]) if ready_text_col else np.nan,
        empty_ratio(ready_df[base_col]) if base_col else np.nan,
        empty_ratio(ready_df[ml_col]) if ml_col else np.nan,
        empty_ratio(ready_df[tr_col]) if tr_col else np.nan,
    ]
})
summary_empty


,Column,Empty_%
0,text,44.16
1,textTranslated,96.18
2,Text,44.16
3,Text_Base,44.26
4,Text_ML,0.00
5,Text_TR,44.26


### **Text length before/after (number of words/letters)**


In [ ]:
def length_stats(series):
    s = series.fillna("").astype(str).str.strip()
    words = s.apply(lambda x: len(x.split()))
    chars = s.str.len()
    return pd.Series({
        "avg_words": round(words.mean(), 2),
        "median_words": float(words.median()),
        "max_words": int(words.max()),
        "avg_chars": round(chars.mean(), 2),
        "median_chars": float(chars.median()),
        "max_chars": int(chars.max())
    })

stats = {}

if raw_text_col:
    stats["Before_raw(text)"] = length_stats(raw_df[raw_text_col])
if raw_tr_col:
    stats["Before_translated(textTranslated)"] = length_stats(raw_df[raw_tr_col])
if ready_text_col:
    stats["After_Text"] = length_stats(ready_df[ready_text_col])
if base_col:
    stats["After_Text_Base"] = length_stats(ready_df[base_col])
if ml_col:
    stats["After_Text_ML"] = length_stats(ready_df[ml_col])
if tr_col:
    stats["After_Text_TR"] = length_stats(ready_df[tr_col])

pd.DataFrame(stats).T


,avg_words,median_words,max_words,avg_chars,median_chars,max_chars
Before_raw(text),5.49,1.0,363.0,31.08,7.0,2243.0
Before_translated(textTranslated),0.36,0.0,277.0,2.08,0.0,1749.0
After_Text,5.43,1.0,363.0,30.76,7.0,2243.0
After_Text_Base,5.38,1.0,366.0,30.13,6.0,2219.0
After_Text_ML,5.83,2.0,321.0,36.09,14.0,2086.0
After_Text_TR,5.36,1.0,366.0,30.06,6.0,2219.0


### **Percentage of Latin letters before/after (verification of Arabic standardization)**

In [ ]:
latin_re = re.compile(r"[A-Za-z]")

def latin_present_ratio(series):
    s = series.fillna("").astype(str)
    has_latin = s.apply(lambda x: bool(latin_re.search(x)))
    return round(has_latin.mean()*100, 2)

latin_report = pd.DataFrame({
    "Column": [raw_text_col, raw_tr_col, ready_text_col, base_col, ml_col, tr_col],
    "Latin_present_%": [
        latin_present_ratio(raw_df[raw_text_col]) if raw_text_col else np.nan,
        latin_present_ratio(raw_df[raw_tr_col]) if raw_tr_col else np.nan,
        latin_present_ratio(ready_df[ready_text_col]) if ready_text_col else np.nan,
        latin_present_ratio(ready_df[base_col]) if base_col else np.nan,
        latin_present_ratio(ready_df[ml_col]) if ml_col else np.nan,
        latin_present_ratio(ready_df[tr_col]) if tr_col else np.nan,
    ]
})
latin_report


,Column,Latin_present_%
0,text,3.60
1,textTranslated,0.06
2,Text,0.20
3,Text_Base,0.19
4,Text_ML,100.00
5,Text_TR,0.00


### **Emoji summary (how many comments it contains + distribution of emotions)**


In [ ]:
emoji_summary = {}

if "Emoji_Count" in ready_df.columns:
    emoji_summary["Reviews_with_emoji_%"] = round((ready_df["Emoji_Count"] > 0).mean()*100, 2)

if "Emoji_Sentiment" in ready_df.columns:
    emoji_summary["Emoji_Sentiment_Distribution"] = ready_df["Emoji_Sentiment"].value_counts(dropna=False)

emoji_summary


{'Reviews_with_emoji_%': 0.79,
 'Emoji_Sentiment_Distribution': Emoji_Sentiment
 NEU    37765
 POS      171
 NEG        2
 Name: count, dtype: int64}

### **Before/After Example Table**


In [ ]:
def text_diff_score(a, b):
    a = "" if pd.isna(a) else str(a)
    b = "" if pd.isna(b) else str(b)
    return abs(len(a) - len(b))

ready_df["Diff_Raw_vs_Base"] = ready_df["Text_Orig"].fillna("").str.len() - ready_df["Text_Base"].fillna("").str.len()
ready_df["Diff_Raw_vs_ML"]   = ready_df["Text_Orig"].fillna("").str.len() - ready_df["Text_ML"].fillna("").str.len()

ready_df[["Diff_Raw_vs_Base", "Diff_Raw_vs_ML"]].describe()


,Diff_Raw_vs_Base,Diff_Raw_vs_ML
count,37938.000000,37938.000000
mean,0.972350,-4.990115
std,6.482819,10.117192
min,-37.000000,-27.000000
25%,0.000000,-8.000000
50%,0.000000,-7.000000
75%,0.000000,-5.000000
max,418.000000,552.000000


In [ ]:
examples_df = (
    ready_df[
        (ready_df["Text_Orig"].notna()) &
        (ready_df["Text_Orig"].str.len() > 20) &
        (
            (ready_df["Diff_Raw_vs_Base"].abs() > 15) |
            (ready_df["Diff_Raw_vs_ML"].abs() > 15) |
            (ready_df["Emoji_Count"] > 0)
        )
    ]
    .sample(10, random_state=42)
)

examples_df.shape


(10, 25)

In [ ]:
comparison_table = pd.DataFrame({
    "Before (Raw Text)": examples_df["Text_Orig"],
    "After (Text_Base)": examples_df["Text_Base"],
    "ML Processing (Text_ML)": examples_df["Text_ML"],
    "Pretrained Processing (Text_TR)": examples_df["Text_TR"],
    "Emoji_Sentiment": examples_df["Emoji_Sentiment"]
})

comparison_table


,Before (Raw Text),After (Text_Base),ML Processing (Text_ML),Pretrained Processing (Text_TR),Emoji_Sentiment
30192,اجمل حديقه بالمملكه❤️,اجمل حديقه بالمملكه,اجمل حديقه بالمملكه EMO_POS,اجمل حديقه بالمملكه,POS
2714,من افضل الاماكن بس ياريت في الصيفيه يعملوا موا...,من افضل الاماكن بس ياريت في الصيفيه يعملوا موا...,افضل الاماكن بس ياريت الصيفيه يعملوا مواقف خار...,من افضل الاماكن بس ياريت في الصيفيه يعملوا موا...,NEU
20183,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,منتزه غابة رغدان حقا مكان يستحق الزيارة للترفي...,NEU
16998,"Amazing experience, a must visit place if you ...",تجربة رائعة، وجهة لا تفوت عند زيارة الباحة منا...,تجربة رائعة، وجهة لا_تفوت زيارة الباحة مناظر ط...,تجربة رائعة، وجهة لا تفوت عند زيارة الباحة منا...,NEU
22259,"Amazing place, there’s many ice cream trucks, ...",مكان رائع، فيه عربات ايس كريم كثيرة، ومطعم واح...,مكان رائع، فيه عربات ايس كريم كثيرة، ومطعم واح...,مكان رائع، فيه عربات ايس كريم كثيرة، ومطعم واح...,NEU
3782,Nice plane with good activities especially zip...,طائرة رائعة مع انشطة ممتعة، لا سيما الانزلاق ب...,طائرة رائعة انشطة ممتعة، لا_سيما الانزلاق بالح...,طائرة رائعة مع انشطة ممتعة، لا سيما الانزلاق ب...,NEU
15852,ك منتزه يعتبر من افضل المنتزهات ف منطقة الباحة...,ك منتزه يعتبر من افضل المنتزهات ف منطقة الباحة...,منتزه يعتبر افضل المنتزهات منطقة الباحة لكن تع...,ك منتزه يعتبر من افضل المنتزهات ف منطقة الباحة...,NEU
3401,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها أ...,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها ا...,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها ا...,غابة جميلة ورايقة وهادئة وتصلح للعوائل وفيها ا...,POS
17540,Very beautiful place to visit in Al Bahah. We ...,مكان جميل جدا للزيارة في الباحة استمتعنا بتجرب...,مكان جميل جدا للزيارة الباحة استمتعنا بتجربة ا...,مكان جميل جدا للزيارة في الباحة استمتعنا بتجرب...,NEU
12512,مكان جميل جداا اعجز عن التعبير عنه صراحة❤️,مكان جميل جداا اعجز عن التعبير عنه صراحة,مكان جميل جداا اعجز التعبير عنه صراحة EMO_POS,مكان جميل جداا اعجز عن التعبير عنه صراحة,POS
